# Module 01 — Lesson 04: Type Hints & Pydantic

This notebook shows how **Python type hints** and **Pydantic** work — the two features that make FastAPI's automatic validation and documentation possible.

Run each cell in order and inspect the output directly inline.

---

## Setup

In [1]:
# Install pydantic if not already present
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "pydantic", "-q"])

from pydantic import BaseModel, Field, field_validator, model_validator
print("✅ Pydantic ready!")

✅ Pydantic ready!


---
## Example 1: Basic Pydantic Model

A Pydantic model is just a class that inherits from `BaseModel`. You declare fields with type hints — Pydantic handles the rest:
- ✅ **Validates** input types
- 🔄 **Coerces** compatible types (e.g. `"25"` → `25`)
- ❌ **Raises** a clear `ValidationError` for bad data

In [2]:
class User(BaseModel):
    name: str
    email: str
    age: int
    is_active: bool = True  # Optional field with default

# ✅ Valid data
user = User(name="Alice", email="alice@example.com", age=30)
print("Model repr :", user)
print("Attribute  :", user.name)
print("Dict       :", user.model_dump())
print("JSON string:", user.model_dump_json())

Model repr : name='Alice' email='alice@example.com' age=30 is_active=True
Attribute  : Alice
Dict       : {'name': 'Alice', 'email': 'alice@example.com', 'age': 30, 'is_active': True}
JSON string: {"name":"Alice","email":"alice@example.com","age":30,"is_active":true}


In [3]:
# 🔄 Auto-coercion — the string "25" is automatically converted to int 25
user2 = User(name="Bob", email="bob@test.com", age="25")
print(f"age value : {user2.age}")
print(f"age type  : {type(user2.age)}")

age value : 25
age type  : <class 'int'>


In [4]:
# ❌ Invalid data — "not_a_number" cannot be converted to int
try:
    bad_user = User(name="Charlie", email="charlie@test.com", age="not_a_number")
except Exception as e:
    print("ValidationError caught:")
    print(e)

ValidationError caught:
1 validation error for User
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not_a_number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


---
## Example 2: Optional and Default Fields

| Field definition | Required? | Default |
|---|---|---|
| `name: str` | ✅ Yes | — |
| `age: int \| None = None` | ❌ No | `None` |
| `bio: str = "No bio"` | ❌ No | `"No bio"` |

In [5]:
class UserProfile(BaseModel):
    name: str                          # Required
    email: str                         # Required
    age: int | None = None             # Optional — defaults to None
    bio: str = "No bio provided"       # Optional — has a default string
    tags: list[str] = []               # Optional — defaults to empty list

# Only supply required fields
profile = UserProfile(name="Alice", email="alice@test.com")
print("With defaults:")
print(profile.model_dump())

With defaults:
{'name': 'Alice', 'email': 'alice@test.com', 'age': None, 'bio': 'No bio provided', 'tags': []}


In [6]:
# Supply all fields
full_profile = UserProfile(
    name="Alice",
    email="alice@test.com",
    age=30,
    bio="Python developer",
    tags=["admin", "verified"]
)
print("Fully populated:")
print(full_profile.model_dump())

Fully populated:
{'name': 'Alice', 'email': 'alice@test.com', 'age': 30, 'bio': 'Python developer', 'tags': ['admin', 'verified']}


---
## Example 3: Nested Models

Pydantic models can contain other models. You can pass a **plain dict** — Pydantic will automatically construct the nested model for you.

In [7]:
class Address(BaseModel):
    street: str
    city: str
    country: str
    zip_code: str

class Employee(BaseModel):
    name: str
    email: str
    department: str
    address: Address          # ← Nested Pydantic model
    skills: list[str] = []

# Pass a plain dict for 'address' — Pydantic converts it automatically
employee = Employee(
    name="Alice",
    email="alice@company.com",
    department="Engineering",
    address={
        "street": "123 Main St",
        "city": "New York",
        "country": "USA",
        "zip_code": "10001"
    },
    skills=["Python", "FastAPI", "Docker"]
)

print("Name       :", employee.name)
print("City       :", employee.address.city)  # Access nested attribute
print("Skills     :", employee.skills)
print()
print("Full dict  :")
print(employee.model_dump())

Name       : Alice
City       : New York
Skills     : ['Python', 'FastAPI', 'Docker']

Full dict  :
{'name': 'Alice', 'email': 'alice@company.com', 'department': 'Engineering', 'address': {'street': '123 Main St', 'city': 'New York', 'country': 'USA', 'zip_code': '10001'}, 'skills': ['Python', 'FastAPI', 'Docker']}


---
## Example 4: Field Validation with Constraints

`Field()` lets you add rules directly on individual fields — no custom code needed:

| Constraint | Applies to | Meaning |
|---|---|---|
| `min_length` / `max_length` | `str` | String length bounds |
| `gt` / `ge` | `int`, `float` | Greater than / greater-or-equal |
| `lt` / `le` | `int`, `float` | Less than / less-or-equal |
| `pattern` | `str` | Must match regex |

In [8]:
class Product(BaseModel):
    name: str     = Field(min_length=1, max_length=100, description="Product name")
    price: float  = Field(gt=0,  description="Price must be > 0")
    quantity: int = Field(ge=0,  le=10000, description="0–10,000 units")
    sku: str      = Field(pattern=r"^[A-Z]{3}-\d{4}$", description="Format: ABC-1234")

# ✅ Valid product
product = Product(name="Laptop", price=999.99, quantity=50, sku="LAP-0001")
print("Valid product:", product)

Valid product: name='Laptop' price=999.99 quantity=50 sku='LAP-0001'


In [9]:
# ❌ Negative price (violates gt=0)
try:
    bad = Product(name="Free Item", price=-5, quantity=1, sku="FRE-0001")
except Exception as e:
    print("Price error:")
    print(e)

Price error:
1 validation error for Product
price
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


In [10]:
# ❌ SKU doesn't match the required regex pattern
try:
    bad_sku = Product(name="Widget", price=10, quantity=1, sku="invalid-sku")
except Exception as e:
    print("SKU error:")
    print(e)

SKU error:
1 validation error for Product
sku
  String should match pattern '^[A-Z]{3}-\d{4}$' [type=string_pattern_mismatch, input_value='invalid-sku', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_pattern_mismatch


---
## Example 5: Custom Validators

When built-in constraints aren't enough, write your own logic:

| Decorator | Validates | When it runs |
|---|---|---|
| `@field_validator("field")` | A single field | After type coercion |
| `@model_validator(mode="after")` | Whole model (cross-field) | After all fields are set |

In [11]:
class UserRegistration(BaseModel):
    username: str      = Field(min_length=3, max_length=30)
    email: str
    password: str      = Field(min_length=8)
    confirm_password: str

    @field_validator("email")
    @classmethod
    def validate_email(cls, v: str) -> str:
        if "@" not in v:
            raise ValueError("Email must contain @")
        if "." not in v.split("@")[1]:
            raise ValueError("Email domain must have a dot")
        return v.lower()          # ← Normalize to lowercase as a side-effect

    @field_validator("username")
    @classmethod
    def validate_username(cls, v: str) -> str:
        if not v.isalnum():
            raise ValueError("Username must be alphanumeric (letters and digits only)")
        return v.lower()          # ← Also normalize

    @model_validator(mode="after")   # Runs after ALL fields are validated
    def passwords_match(self):
        if self.password != self.confirm_password:
            raise ValueError("Passwords do not match")
        return self

# ✅ Valid registration — note mixed-case inputs get normalised
reg = UserRegistration(
    username="Alice123",
    email="Alice@Example.COM",
    password="securepass123",
    confirm_password="securepass123"
)
print("username :", reg.username)  # alice123  ← lowercased
print("email    :", reg.email)     # alice@example.com  ← lowercased

username : alice123
email    : alice@example.com


In [12]:
# ❌ Passwords don't match — caught by @model_validator
try:
    bad_reg = UserRegistration(
        username="bob",
        email="bob@test.com",
        password="password123",
        confirm_password="different456"
    )
except Exception as e:
    print("Password mismatch error:")
    print(e)

Password mismatch error:
1 validation error for UserRegistration
  Value error, Passwords do not match [type=value_error, input_value={'username': 'bob', 'emai...ssword': 'different456'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [13]:
# ❌ Invalid username — contains a hyphen
try:
    bad_username = UserRegistration(
        username="bad-name!",
        email="test@test.com",
        password="password123",
        confirm_password="password123"
    )
except Exception as e:
    print("Username error:")
    print(e)

Username error:
1 validation error for UserRegistration
username
  Value error, Username must be alphanumeric (letters and digits only) [type=value_error, input_value='bad-name!', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


---
## Example 6: Model Inheritance (The FastAPI Pattern)

The most important pattern in real FastAPI apps — use **inheritance** to share fields and avoid repeating yourself:

```
UserBase          ← shared fields (name, email)
├── UserCreate    ← adds 'password'   (used as request body)
└── UserResponse  ← adds 'id'         (used as response — no password!)
```

This prevents passwords from ever leaking into API responses.

In [14]:
class UserBase(BaseModel):
    """Shared fields — reused by every user schema."""
    name: str
    email: str

class UserCreate(UserBase):
    """Client → Server: includes the raw password."""
    password: str

class UserResponse(UserBase):
    """Server → Client: includes id, NO password."""
    id: int
    is_active: bool

# --- Simulate what FastAPI does internally ---

# 1. Client sends this
create_data = UserCreate(name="Alice", email="alice@test.com", password="secret123")
print("UserCreate (what the client sends):")
print(create_data.model_dump())
print()

UserCreate (what the client sends):
{'name': 'Alice', 'email': 'alice@test.com', 'password': 'secret123'}



In [15]:
# 2. After saving to the DB, build the response — password is gone!
response_data = UserResponse(
    id=1,
    name=create_data.name,
    email=create_data.email,
    is_active=True
)
print("UserResponse (what the API returns — no password!):")
print(response_data.model_dump())
print()
print("Has 'password' field?", hasattr(response_data, 'password'))  # False ✅

UserResponse (what the API returns — no password!):
{'name': 'Alice', 'email': 'alice@test.com', 'id': 1, 'is_active': True}

Has 'password' field? False


---
## Summary

| Feature | How |
|---|---|
| Define a model | `class X(BaseModel): field: type` |
| Optional field | `field: type \| None = None` |
| Field constraints | `field: type = Field(gt=0, min_length=3)` |
| Nested model | `address: Address` — pass a dict, Pydantic converts |
| Custom field validator | `@field_validator("field") @classmethod def f(cls, v)` |
| Cross-field validator | `@model_validator(mode="after") def f(self)` |
| Prevent password leaks | Separate `UserCreate` / `UserResponse` models |

---
## Practice Exercises

Try these in the cell below:

1. Add a `phone` field to `UserProfile` that matches a regex like `^\+?[0-9\s-]{7,15}$`.
2. Create a `BlogPost` model with: `title` (5–200 chars), `content` (required), `tags` (list of strings, default empty), `published` (bool, default `False`).
3. Add a `@model_validator` that ensures `published` is `False` when `content` is shorter than 50 characters.

In [ ]:
# Your practice code here ✍️
